# Lesson 4.3 — Enforcing Structured Output with tool_use & JSON Schemas

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Use tool_use with JSON schemas for guaranteed schema-compliant output
- Distinguish tool_choice auto, any, and forced for extraction
- Design required vs optional/nullable fields and enum+other patterns
- Recognize that strict schemas remove syntax errors but not semantic errors

**You will produce:** An extraction tool schema (nullable + enum/other) and tool_choice configs for extraction.

## Concepts

**tool_use with JSON schemas is the most reliable way** to get guaranteed schema-compliant structured output - it eliminates JSON syntax errors. But it does **not** prevent semantic errors (line items that don't sum to total, values in the wrong field) - that's Lesson 4.4.

**tool_choice:**
- `auto` - model may return text instead of a tool call.
- `any` - must call a tool, can choose which (use when multiple extraction schemas exist and the document type is unknown).
- forced `{"type": "tool", "name": "extract_metadata"}` - run a specific extraction first, then enrich in follow-up turns.

**Schema design:** make fields **optional/nullable** when the source may not contain them (prevents fabrication to satisfy required fields); add enum values like **`"unclear"`** for ambiguity and **`"other"` + detail** for extensible categories; include **format normalization rules** in the prompt alongside the strict schema.

> **Reference:** [Anthropic - tool use for structured output](https://docs.anthropic.com/en/docs/build-with-claude/tool-use)

In [ ]:
# !pip install anthropic
import os, json

try:
    import anthropic
except ImportError:
    anthropic = None
    print("anthropic SDK not installed. Run: pip install anthropic")

API_KEY = os.environ.get("ANTHROPIC_API_KEY")
HAVE_API = bool(API_KEY) and anthropic is not None

client = anthropic.Anthropic() if HAVE_API else None  # reads ANTHROPIC_API_KEY

# Current model aliases (confirm latest at https://docs.anthropic.com/en/docs/about-claude/models)
MODEL_HAIKU = "claude-haiku-4-5"
MODEL_SONNET = "claude-sonnet-4-5"
MODEL_OPUS = "claude-opus-4-1"

def mock_text(prompt):
    """Offline fallback so the notebook runs without an API key."""
    return f"[MOCK RESPONSE] Set ANTHROPIC_API_KEY for a live call. Prompt was: {str(prompt)[:80]}..."

print("API ready" if HAVE_API else "No API key - cells will print mock responses")

## Code-Along: Step 1 — Define an extraction tool with nullable + enum/other

Optional/nullable fields and an 'other'+detail enum prevent fabrication.

In [ ]:
extract_invoice = {
    "name": "extract_invoice",
    "description": "Extract invoice fields. Return null for absent fields; do not guess.",
    "input_schema": {
        "type": "object",
        "properties": {
            "invoice_number": {"type": ["string", "null"]},
            "total": {"type": ["number", "null"]},
            "currency": {"type": "string",
                "enum": ["USD", "EUR", "GBP", "other", "unclear"]},
            "currency_other_detail": {"type": ["string", "null"]},
            "line_items": {"type": "array", "items": {"type": "object"}},
        },
        "required": ["currency"],
    },
}
print(extract_invoice['input_schema']['properties']['currency']['enum'])

## Code-Along: Step 2 — Choose tool_choice for extraction

Unknown doc type -> any; must run metadata first -> forced.

In [ ]:
def tool_choice(mode, name=None):
    if mode == "any":
        return {"type": "any"}
    if mode == "forced":
        return {"type": "tool", "name": name}
    return {"type": "auto"}

print("unknown document type:", tool_choice("any"))
print("metadata first:", tool_choice("forced", "extract_metadata"))

## Code-Along: Step 3 — Call the model with forced extraction (guarded)

With a key, we force the tool; offline we mock a compliant tool_use payload.

In [ ]:
doc = "Invoice 0098 total CA$120.00 for consulting"
if HAVE_API:
    resp = client.messages.create(model=MODEL_SONNET, max_tokens=512,
        tools=[extract_invoice], tool_choice=tool_choice('forced', 'extract_invoice'),
        messages=[{"role": "user", "content": doc}])
    tu = next((b for b in resp.content if b.type == 'tool_use'), None)
    print(tu.input if tu else '[no tool_use]')
else:
    print({"invoice_number": "0098", "total": 120.0,
           "currency": "other", "currency_other_detail": "CAD", "line_items": []})

## Exercise

Extend the schema with an optional `due_date` (nullable) and a `document_type` enum including `"unclear"`. Process a document missing the due date and assert the extraction returns `None` rather than a fabricated date. Then pick the correct tool_choice when you have three extraction schemas and an unknown document type.

In [ ]:
# TODO: add due_date (nullable) and document_type enum with 'unclear'
# TODO: assert extracted['due_date'] is None for a doc with no date
# TODO: choose tool_choice for unknown-type/multi-schema extraction

## Key Takeaways

- tool_use + JSON schema guarantees schema-compliant output and removes syntax errors.
- Strict schemas do NOT prevent semantic errors - validate separately (Lesson 4.4).
- tool_choice: auto/any/forced - use 'any' for unknown types, forced to run a tool first.
- Make absent fields nullable and add enum 'unclear'/'other'+detail to prevent fabrication.

---

**Next:** [Lesson 4.4 — Validation, Retry & Feedback Loops](./lesson-4.4-validation-retry-loops.ipynb)